<a href="https://colab.research.google.com/github/Bayronguerrero/Parcial04_Bayron_lopez_2509672020/blob/main/notebook/Asociacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from itertools import combinations


In [2]:
# 1. Cargar datos
url= "https://raw.githubusercontent.com/Bayronguerrero/Parcial04_Bayron_lopez_2509672020/refs/heads/main/archivo/clave_H_asociacion.csv"
df=pd.read_csv(url)
df.head()


,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,H-T0001,H-C0026,2026-04-10,Cloud,Backup_cloud,2,Web
1,H-T0001,H-C0026,2026-04-10,Productividad,CRM,2,Web
2,H-T0001,H-C0026,2026-04-10,Educacion,Certificacion,1,Web
3,H-T0001,H-C0026,2026-04-10,Productividad,Notas,1,Web
4,H-T0002,H-C0038,2026-03-16,Productividad,CRM,1,Tienda


El dataset cargado contiene información de transacciones, con las siguientes columnas:

*   `transaccion_id`: Identificador único de cada transacción.
*   `cliente_id`: Identificador del cliente que realizó la transacción.
*   `fecha`: Fecha en que se realizó la transacción.
*   `categoria`: Categoría del producto/servicio.
*   `item`: Nombre específico del producto/servicio adquirido.
*   `cantidad`: Cantidad del `item` comprado.
*   `canal`: Canal de venta (e.g., Web, Tienda).

In [4]:
# 2. Verificar valores nulos, duplicados y tipos de datos
print("--- Información del DataFrame ---")
df.info()

print("\n--- Valores nulos por columna ---")
print(df.isnull().sum())

print("\n--- Filas duplicadas ---")
print(f"Número de filas duplicadas: {df.duplicated().sum()}")

--- Información del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 624 entries, 0 to 623
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  624 non-null    object
 1   cliente_id      624 non-null    object
 2   fecha           624 non-null    object
 3   categoria       624 non-null    object
 4   item            624 non-null    object
 5   cantidad        624 non-null    int64 
 6   canal           623 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.3+ KB

--- Valores nulos por columna ---
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

--- Filas duplicadas ---
Número de filas duplicadas: 1


In [5]:
# 3. Limpiar y preparar los datos para reglas de asociación

# Eliminar filas duplicadas
df.drop_duplicates(inplace=True)
print(f"Número de filas duplicadas después de la eliminación: {df.duplicated().sum()}")

# Eliminar filas con valores nulos (específicamente en 'canal')
df.dropna(inplace=True)
print(f"Valores nulos por columna después de la eliminación:\n{df.isnull().sum()}")

# Convertir 'fecha' a tipo datetime (opcional, pero buena práctica)
df['fecha'] = pd.to_datetime(df['fecha'])

# Agrupar items por transaccion_id para formar las transacciones
transacciones = df.groupby('transaccion_id')['item'].apply(list).reset_index()

# Mostrar las primeras transacciones para verificar
print("\n--- Primeras 5 transacciones ---")
print(transacciones.head())

# Preparar los datos para el algoritmo Apriori (One-Hot Encoding)
# Crear una lista de todas las transacciones (lista de listas de items)
lista_transacciones = transacciones['item'].tolist()

# Usar TransactionEncoder de mlxtend para convertir la lista de transacciones en un DataFrame one-hot encoded
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(lista_transacciones).transform(lista_transacciones)
df_items = pd.DataFrame(te_ary, columns=te.columns_)

print("\n--- DataFrame One-Hot Encoded (primeras 5 filas) ---")
print(df_items.head())

Número de filas duplicadas después de la eliminación: 0
Valores nulos por columna después de la eliminación:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             0
dtype: int64

--- Primeras 5 transacciones ---
  transaccion_id                                               item
0        H-T0001          [Backup_cloud, CRM, Certificacion, Notas]
1        H-T0002                 [CRM, Certificacion, Curso_ingles]
2        H-T0003   [Almacenamiento, Backup_cloud, Dominio, Hosting]
3        H-T0004  [Almacenamiento, Gestor_password, Plan_deporte...
4        H-T0005             [Almacenamiento, Gestor_password, VPN]

--- DataFrame One-Hot Encoded (primeras 5 filas) ---
   Agenda  Almacenamiento  Antivirus  Backup_cloud    CRM  Certificacion  \
0   False           False      False          True   True           True   
1   False           False      False         False   True           True   
2   False     

In [6]:
# 4. Aplicar el algoritmo Apriori y generar reglas de asociación
from mlxtend.frequent_patterns import apriori, association_rules

# Aplicar Apriori para encontrar los itemsets frecuentes
# Se puede ajustar el min_support según la granularidad deseada
frequent_itemsets = apriori(df_items, min_support=0.05, use_colnames=True)

print("\n--- Itemsets Frecuentes (primeras 5) ---")
print(frequent_itemsets.head())

# Generar reglas de asociación
# Se puede ajustar el min_confidence
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Mostrar las 10 reglas más relevantes, ordenadas por lift descendente
print("\n--- Las 10 reglas de asociación más relevantes (ordenadas por lift) ---")
rules_sorted = rules.sort_values(by="lift", ascending=False)
print(rules_sorted.head(10))


--- Itemsets Frecuentes (primeras 5) ---
   support          itemsets
0    0.095          (Agenda)
1    0.175  (Almacenamiento)
2    0.080       (Antivirus)
3    0.115    (Backup_cloud)
4    0.115             (CRM)

--- Las 10 reglas de asociación más relevantes (ordenadas por lift) ---
          antecedents        consequents  antecedent support  \
15  (Gestor_password)              (VPN)               0.195   
14              (VPN)  (Gestor_password)               0.245   
12          (Dominio)          (Hosting)               0.185   
13          (Hosting)          (Dominio)               0.225   
17       (Plan_video)    (Plan_familiar)               0.265   
16    (Plan_familiar)       (Plan_video)               0.215   
7     (Certificacion)     (Curso_python)               0.255   
6      (Curso_python)    (Certificacion)               0.235   
4     (Certificacion)              (CRM)               0.255   
5               (CRM)    (Certificacion)               0.115   

    co

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 5. Interpretación de las 5 Reglas de Asociación más Relevantes (ordenadas por Lift)

Las reglas de asociación nos muestran qué productos o servicios tienden a ser comprados juntos. El `lift` es una métrica clave, indicando cuánto más probable es que se compre el consecuente cuando se compra el antecedente, en comparación con la probabilidad de comprar el consecuente por sí solo. Un `lift` mayor a 1 sugiere una asociación positiva.

Aquí la interpretación de las 5 reglas con mayor `lift`:

1.  **Regla:** `{Gestor_password} -> {VPN}` (Lift: 2.62)
    *   **Interpretación:** Los clientes que adquieren un `Gestor_password` son 2.62 veces más propensos a contratar también un servicio de `VPN` que un cliente promedio. Esto indica una fuerte conciencia de seguridad entre estos usuarios, buscando proteger tanto sus credenciales como su conexión a internet.

2.  **Regla:** `{VPN} -> {Gestor_password}` (Lift: 2.62)
    *   **Interpretación:** Inversamente, los clientes que contratan un servicio de `VPN` son 2.62 veces más propensos a adquirir un `Gestor_password`. Esta regla refuerza la anterior, mostrando una relación bidireccional fuerte entre estos dos productos de seguridad.

3.  **Regla:** `{Dominio} -> {Hosting}` (Lift: 2.52)
    *   **Interpretación:** Los clientes que compran un `Dominio` (nombre de internet) son 2.52 veces más propensos a contratar un servicio de `Hosting` (alojamiento web). Esta es una asociación muy natural, ya que la mayoría de los sitios web requieren ambos para funcionar.

4.  **Regla:** `{Hosting} -> {Dominio}` (Lift: 2.52)
    *   **Interpretación:** Los clientes que contratan `Hosting` son 2.52 veces más propensos a comprar un `Dominio`. Similar a la anterior, confirma la fuerte complementariedad de estos dos servicios esenciales para cualquier presencia en línea.

5.  **Regla:** `{Plan_video} -> {Plan_familiar}` (Lift: 2.46)
    *   **Interpretación:** Los clientes que adquieren un `Plan_video` (posiblemente un servicio de streaming o contenido multimedia) son 2.46 veces más propensos a tener también un `Plan_familiar`. Esto sugiere que el consumo de video es una actividad común en hogares con planes familiares o que los clientes de planes familiares buscan opciones de entretenimiento en video.

### 6. Propuestas de Recomendaciones Comerciales

Basándonos en las reglas de asociación encontradas, podemos formular las siguientes recomendaciones comerciales:

1.  **Paquete de Seguridad Digital Integral:** Crear una oferta de paquete que incluya `Gestor_password` y `VPN` con un precio preferencial. Esto atraería a clientes preocupados por la seguridad que ya están comprando uno de estos productos, incentivándolos a adquirir el otro. Se puede promocionar como una "Solución Completa para tu Seguridad Online".

2.  **Oferta de Lanzamiento Web "Todo en Uno":** Desarrollar un paquete de inicio para nuevos clientes o emprendedores que deseen tener presencia en línea, incluyendo `Dominio` y `Hosting` a un costo reducido. Esta estrategia capitaliza la fuerte relación entre ambos productos, simplificando la decisión de compra para el cliente y aumentando el valor promedio de la transacción.

3.  **Promoción Cruzada de Contenido y Planes Familiares:** Al momento de que un cliente adquiera un `Plan_video`, ofrecerle automáticamente un descuento o una mejora en un `Plan_familiar`, o viceversa. Esto puede ser presentado como una "Experiencia Familiar Mejorada" que combine entretenimiento con los beneficios de un plan multi-usuario, fomentando así la retención y el cross-selling entre segmentos de clientes.